# Concept Recovery Analysis

Pipeline for a single concept × all three methods:
1. Extracts CLIP CLS activations at `DEBIAS_LAYER` (no prior debiasing).
2. Iteratively (`N_ITER` times) computes CAV and projects it out. Convention: `iter k` = state after k projections.
3. Injects the finally debiased activations back into CLIP at `DEBIAS_LAYER`.
4. The rest of the network runs without debiasing — CLS tokens are collected from layers `DEBIAS_LAYER+1 .. NUM_LAYERS-1`.
5. At each of those layers a LR (CV over C) is trained — the "concept recovery" curve.

**Change only `CONCEPT`** and optionally `DEBIAS_LAYER` / `N_ITER` — everything else follows automatically.

Outputs:
- `data/multiple_debiasing/{CONCEPT}/{method}/recovery/layer{DEBIAS_LAYER:02d}_{method}_iter{N_ITER}/iter_debiasing_accuracy.csv`
- `data/multiple_debiasing/{CONCEPT}/{method}/recovery/layer{DEBIAS_LAYER:02d}_{method}_iter{N_ITER}/layer_recovery_accuracy.csv`
- `notebooks/results/multiple_debias/{CONCEPT}/recovery/recovery_layer{DEBIAS_LAYER:02d}_iter{N_ITER}_{CONCEPT}.png`
- `notebooks/results/multiple_debias/{CONCEPT}/recovery/recovery_summary_layer{DEBIAS_LAYER:02d}_iter{N_ITER}.csv`

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from dotenv import load_dotenv
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
load_dotenv()

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.torch_lr import TorchLR
from software.viz import plot_recovery

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT       = 'eyeglasses'
METHODS       = ['diff_means', 'lr', 'pclarc']
DEBIAS_LAYER  = 15
N_ITER        = 7

NUM_LAYERS     = 24
GPU_BATCH_SIZE = 32
NUM_WORKERS    = 0
MODEL_ID       = 'openai/clip-vit-large-patch14'

# Solver for logistic regression (both CAV training and recovery probing):
#   'torch_lr' — TorchLR (LBFGS on GPU, stable, recommended)
#   'sgd'      — SGDClassifier from sklearn (stochastic gradient descent)
SOLVER   = 'torch_lr'
SOLVER_C = 0.1

RECOVERY_LAYERS = list(range(DEBIAS_LAYER + 1, NUM_LAYERS))

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR    = ROOT / 'data' / 'images'
DATA_OUT_BASE = ROOT / 'data' / 'multiple_debiasing' / CONCEPT
PLOT_DIR      = ROOT / 'notebooks' / 'results' / 'multiple_debias' / CONCEPT / 'recovery'

assert 0 <= DEBIAS_LAYER < NUM_LAYERS - 1, \
    f'DEBIAS_LAYER must be in [0, {NUM_LAYERS - 2}]'
assert METADATA_PATH.exists(), f'Missing: {METADATA_PATH}'
assert IMAGES_DIR.exists(),    f'Missing: {IMAGES_DIR}'

df_meta  = pd.read_csv(METADATA_PATH)
assert CONCEPT in df_meta.columns, f'Column "{CONCEPT}" not in metadata.csv'
df_train = df_meta[df_meta['split'] == 'train'].reset_index(drop=True)
df_test  = df_meta[df_meta['split'] == 'test'].reset_index(drop=True)

run_id_str = f'layer{DEBIAS_LAYER:02d}_iter{N_ITER}'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
for m in METHODS:
    (DATA_OUT_BASE / m / 'recovery' / run_id_str).mkdir(parents=True, exist_ok=True)

print(f'Concept         : {CONCEPT}')
print(f'Solver          : {SOLVER} (C={SOLVER_C})')
print(f'Methods         : {METHODS}')
print(f'Debias layer    : {DEBIAS_LAYER}  |  Iterations: {N_ITER}')
print(f'Recovery layers : {RECOVERY_LAYERS}')
print(f'Train           : {len(df_train)} | Test: {len(df_test)}')

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype    = torch.bfloat16 if device == 'cuda' else torch.float32

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_ID, dtype=dtype, low_cpu_mem_usage=True, token=HF_TOKEN,
).to(device).eval()
print(f'Model: {MODEL_ID} | {device} | {dtype}')


class CelebADataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df         = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.processor  = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.images_dir / row['filename']).convert('RGB')
        px  = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        return px, int(row[CONCEPT])


def make_loader(df):
    return DataLoader(
        CelebADataset(df, IMAGES_DIR, processor),
        batch_size=GPU_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )


loader_train = make_loader(df_train)
loader_test  = make_loader(df_test)

In [ ]:
def make_cav_clf():
    """LR classifier for CAV training (no CV — C fixed by SOLVER_C)."""
    if SOLVER == 'torch_lr':
        return TorchLR(C=SOLVER_C, max_iter=500, random_state=42)
    return SGDClassifier(
        loss='log_loss', penalty='l2',
        alpha=1.0 / (2.0 * SOLVER_C),
        max_iter=1000, tol=1e-4, random_state=42,
    )


def _validate_xy(X_tr, y_tr, where):
    if not np.isfinite(X_tr).all():
        n_bad = int((~np.isfinite(X_tr)).sum())
        raise ValueError(
            f'{where}: X_tr contains {n_bad} NaN/Inf values '
            f'(likely numerical accumulation across iterations).'
        )
    n_pos = int((y_tr == 1).sum())
    n_neg = int((y_tr == 0).sum())
    if n_pos == 0 or n_neg == 0:
        raise ValueError(
            f'{where}: one class is empty (pos={n_pos}, neg={n_neg}).'
        )


def compute_diff_means(X_tr, y_tr):
    _validate_xy(X_tr, y_tr, 'compute_diff_means')
    X    = X_tr.astype(np.float64)
    diff = X[y_tr == 1].mean(axis=0) - X[y_tr == 0].mean(axis=0)
    n    = np.linalg.norm(diff)
    if n < 1e-10:
        raise ValueError('compute_diff_means: class means nearly identical.')
    return (diff / n).astype(np.float32)


def compute_lr_cav(X_tr, y_tr):
    _validate_xy(X_tr, y_tr, 'compute_lr_cav')
    clf = make_cav_clf()
    clf.fit(X_tr, y_tr)
    w = clf.coef_[0].astype(np.float64)
    n = np.linalg.norm(w)
    if n < 1e-10:
        raise ValueError('compute_lr_cav: weights have zero norm.')
    return (w / n).astype(np.float32), clf


def fit_dm_threshold(X_tr, y_tr, cav):
    proj = X_tr.astype(np.float64) @ cav.astype(np.float64)
    return float((proj[y_tr == 1].mean() + proj[y_tr == 0].mean()) / 2)


def cav_accuracy_dm(X, y, cav, threshold):
    proj = X.astype(np.float64) @ cav.astype(np.float64)
    return accuracy_score(y, (proj > threshold).astype(int))


def cav_accuracy_lr(X, y, clf):
    return clf.score(X, y)


def apply_orthogonal(X, cav):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    return (X64 - np.outer(X64 @ c64, c64)).astype(np.float32)


def apply_pclarc(X, cav, target_val):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    return (X64 + np.outer(target_val - X64 @ c64, c64)).astype(np.float32)

In [ ]:
class _StopForward(Exception):
    pass


def extract_layer(loader, layer_idx):
    """Returns (X: float32 (N, D), y: int (N,)) — CLS token from `layer_idx`."""
    _buf = []

    def _hook(module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        _buf.append(hidden[:, 0, :].detach().float().cpu().numpy())
        raise _StopForward()

    handle = model.vision_model.encoder.layers[layer_idx].register_forward_hook(_hook)
    X_all, y_all = [], []
    try:
        with torch.no_grad():
            for pixels, labels in loader:
                _buf.clear()
                try:
                    model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
                except _StopForward:
                    pass
                if _buf:
                    X_all.append(_buf[0])
                y_all.extend(labels.tolist())
    finally:
        handle.remove()

    return np.concatenate(X_all, axis=0), np.array(y_all, dtype=int)


def run_with_injection(loader, inject_layer, debiased_cls, recovery_layers):
    """Forward pass: replaces CLS at `inject_layer` with `debiased_cls`,
    collects CLS from each layer in `recovery_layers`. Returns dict {layer -> (N, D)}."""
    encoder_layers = model.vision_model.encoder.layers
    captures = {k: [] for k in recovery_layers}
    ptr = [0]

    def inject_hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        hidden   = output[0] if is_tuple else output
        B        = hidden.shape[0]
        cls_deb  = torch.from_numpy(
            debiased_cls[ptr[0]:ptr[0] + B]
        ).to(hidden.device, dtype=hidden.dtype)
        ptr[0] += B
        h_new = hidden.clone()
        h_new[:, 0, :] = cls_deb
        return (h_new,) + output[1:] if is_tuple else h_new

    def make_cap(k):
        def hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            captures[k].append(hidden[:, 0, :].detach().float().cpu().numpy())
        return hook

    handles = [encoder_layers[inject_layer].register_forward_hook(inject_hook)]
    for k in recovery_layers:
        handles.append(encoder_layers[k].register_forward_hook(make_cap(k)))

    try:
        with torch.no_grad():
            for pixels, _ in loader:
                model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
    finally:
        for h in handles:
            h.remove()

    return {k: np.concatenate(v, axis=0) for k, v in captures.items()}

In [ ]:
def iter_debias(X_tr_raw, y_tr, X_te_raw, y_te, method, n_iter):
    """Iterative debiasing. Returns (X_tr_deb, X_te_deb, df_iter)."""
    X_tr_deb = X_tr_raw.copy()
    X_te_deb = X_te_raw.copy()

    iter_records = []
    prev_cav, prev_target = None, None

    for it in range(n_iter + 1):
        if it > 0:
            if method == 'pclarc':
                X_tr_deb = apply_pclarc(X_tr_deb, prev_cav, prev_target)
                X_te_deb = apply_pclarc(X_te_deb, prev_cav, prev_target)
            else:
                X_tr_deb = apply_orthogonal(X_tr_deb, prev_cav)
                X_te_deb = apply_orthogonal(X_te_deb, prev_cav)

        if method == 'lr':
            cav, clf = compute_lr_cav(X_tr_deb, y_tr)
            acc_tr = cav_accuracy_lr(X_tr_deb, y_tr, clf)
            acc_te = cav_accuracy_lr(X_te_deb, y_te, clf)
        else:
            cav       = compute_diff_means(X_tr_deb, y_tr)
            threshold = fit_dm_threshold(X_tr_deb, y_tr, cav)
            acc_tr    = cav_accuracy_dm(X_tr_deb, y_tr, cav, threshold)
            acc_te    = cav_accuracy_dm(X_te_deb, y_te, cav, threshold)

        if method == 'pclarc':
            prev_target = float(
                (X_tr_deb.astype(np.float64)[y_tr == 0] @ cav.astype(np.float64)).mean()
            )
        else:
            prev_target = None
        prev_cav = cav

        iter_records.append({'iteration': it, 'train_acc': acc_tr, 'test_acc': acc_te})

    return X_tr_deb, X_te_deb, pd.DataFrame(iter_records)


def train_recovery(caps_tr, y_tr, caps_te, y_te, recovery_layers):
    """Trains LR on CLS tokens from each recovery layer. Returns DataFrame."""
    rows = []
    for k in recovery_layers:
        X_tr_k, X_te_k = caps_tr[k], caps_te[k]
        _validate_xy(X_tr_k, y_tr, f'recovery L{k}')
        clf = make_cav_clf()
        clf.fit(X_tr_k, y_tr)
        rows.append({
            'layer_id':  k,
            'train_acc': clf.score(X_tr_k, y_tr),
            'test_acc':  clf.score(X_te_k, y_te),
        })
    return pd.DataFrame(rows)

In [ ]:
all_results = {}  # {method: {'iter': df, 'recovery': df}}

# Extract activations once — shared across all methods
print(f'Extracting layer {DEBIAS_LAYER} activations...')
X_tr_raw, y_tr = extract_layer(loader_train, DEBIAS_LAYER)
X_te_raw, y_te = extract_layer(loader_test,  DEBIAS_LAYER)
print(f'  train: {X_tr_raw.shape} | test: {X_te_raw.shape}')

for method in tqdm(METHODS, desc='Methods'):
    data_out = DATA_OUT_BASE / method / 'recovery' / run_id_str

    # 1) Iterative debiasing
    X_tr_deb, X_te_deb, df_iter = iter_debias(
        X_tr_raw, y_tr, X_te_raw, y_te, method, N_ITER
    )
    df_iter.to_csv(data_out / 'iter_debiasing_accuracy.csv', index=False)

    # 2) Forward pass with debiased CLS injected at DEBIAS_LAYER
    caps_tr = run_with_injection(loader_train, DEBIAS_LAYER, X_tr_deb, RECOVERY_LAYERS)
    caps_te = run_with_injection(loader_test,  DEBIAS_LAYER, X_te_deb, RECOVERY_LAYERS)

    # 3) Recovery: LR (CV over C) per recovery layer
    df_rec = train_recovery(caps_tr, y_tr, caps_te, y_te, RECOVERY_LAYERS)
    df_rec.to_csv(data_out / 'layer_recovery_accuracy.csv', index=False)

    all_results[method] = {'iter': df_iter, 'recovery': df_rec}

    baseline = df_iter.iloc[0]['test_acc']
    after    = df_iter[df_iter['iteration'] == N_ITER]['test_acc'].values[0]
    max_rec  = df_rec['test_acc'].max()
    print(
        f'  [{method:10s}] test: baseline={baseline:.3f} -> '
        f'after {N_ITER} iter={after:.3f} | max recovery={max_rec:.3f}'
    )

print('\nPipeline done.')

## Visualization

In [ ]:
plot_recovery(
    all_results, CONCEPT, DEBIAS_LAYER, N_ITER,
    save_path=PLOT_DIR / f'recovery_{run_id_str}_{CONCEPT}.png',
)

## Summary table

In [ ]:
summary_rows = []
for method, d in all_results.items():
    df_iter = d['iter']
    df_rec  = d['recovery']
    baseline  = float(df_iter.iloc[0]['test_acc'])
    after     = float(df_iter[df_iter['iteration'] == N_ITER]['test_acc'].values[0])
    max_rec   = float(df_rec['test_acc'].max())
    max_layer = int(df_rec.loc[df_rec['test_acc'].idxmax(), 'layer_id'])
    summary_rows.append({
        'concept':                  CONCEPT,
        'method':                   method,
        'debias_layer':             DEBIAS_LAYER,
        'n_iter':                   N_ITER,
        'baseline_test':            round(baseline, 3),
        f'after_{N_ITER}iter_test': round(after, 3),
        'max_recovery_test':        round(max_rec, 3),
        'max_recovery_layer':       max_layer,
    })

df_summary = pd.DataFrame(summary_rows)
summary_path = PLOT_DIR / f'recovery_summary_{run_id_str}.csv'
df_summary.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}\n')
print(df_summary.to_string(index=False))